# Autoencoders y VAE con PyTorch

Implementaremos un autoencoder denso, uno convolucional y un autoencoder variacional. Los tres comprimen y reconstruyen imágenes de Fashion MNIST.

## Entorno IPD434

Selecciona el kernel **IPD434**. El notebook usa CUDA si está disponible. Fashion MNIST se lee desde `../data/fashion_mnist` y se descarga solo si aún no está en esa carpeta.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

## Preparar los datos

El objetivo es reconstruir la imagen de entrada, por lo que no necesitamos las etiquetas de las prendas. Apartamos validación para seguir la pérdida y una prueba pequeña para observar reconstrucciones finales.

In [ ]:
DATA_ROOT = Path('../data/fashion_mnist')
dataset = datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=transforms.ToTensor())
test_full = datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=transforms.ToTensor())
generator = torch.Generator().manual_seed(SEED)
train_set, val_set, unused = random_split(dataset, [8000, 2000, len(dataset) - 10000], generator=generator)
test_set, unused_test = random_split(test_full, [2000, len(test_full) - 2000], generator=torch.Generator().manual_seed(SEED + 1))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(SEED), num_workers=0)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)
print('Tamaños:', len(train_set), len(val_set), len(test_set))

## Autoencoder denso

El encoder reduce cada imagen a un código de 30 valores. El decoder intenta reconstruir sus 784 píxeles. La pérdida binaria compara la imagen reconstruida con la original.

In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, latent_dim=30):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(28 * 28, 100), nn.SELU(),
            nn.Linear(100, latent_dim), nn.SELU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 100), nn.SELU(),
            nn.Linear(100, 28 * 28), nn.Sigmoid(),
        )

    def forward(self, images):
        code = self.encoder(images)
        reconstruction = self.decoder(code)
        return reconstruction.view(-1, 1, 28, 28)

def fit_autoencoder(model, epochs=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {'train': [], 'validation': []}
    for epoch in range(1, epochs + 1):
        model.train()
        total, count = 0.0, 0
        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad(set_to_none=True)
            reconstruction = model(images)
            loss = F.binary_cross_entropy(reconstruction, images)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(images)
            count += len(images)
        history['train'].append(total / count)
        model.eval()
        val_total, val_count = 0.0, 0
        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                val_total += F.binary_cross_entropy(model(images), images, reduction='sum').item()
                val_count += images.numel()
        history['validation'].append(val_total / val_count)
        print(f'Época {epoch}/{epochs} | train={history["train"][-1]:.4f} | val={history["validation"][-1]:.4f}')
    return history

dense_ae = DenseAutoencoder()
dense_history = fit_autoencoder(dense_ae, epochs=5)

In [ ]:
images, _ = next(iter(test_loader))
dense_ae.eval()
with torch.no_grad():
    reconstructions = dense_ae(images.to(device)).cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i in range(8):
    axes[0, i].imshow(images[i, 0], cmap='gray')
    axes[1, i].imshow(reconstructions[i, 0], cmap='gray')
    axes[0, i].axis('off'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original'); axes[1, 0].set_ylabel('Reconstruida')
plt.tight_layout(); plt.show()

## Autoencoder convolucional

Las convoluciones reducen el tamaño espacial y aumentan los canales. El decoder recupera la resolución mediante convoluciones transpuestas. Aquí entrenamos con imágenes limpias como entrada y objetivo, igual que en el ejemplo original.

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.SELU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.SELU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.SELU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def forward(self, images):
        return self.decoder(self.encoder(images))

conv_ae = ConvAutoencoder()
conv_history = fit_autoencoder(conv_ae, epochs=5)

In [ ]:
conv_ae.eval()
with torch.no_grad():
    conv_reconstructions = conv_ae(images.to(device)).cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i in range(8):
    axes[0, i].imshow(images[i, 0], cmap='gray')
    axes[1, i].imshow(conv_reconstructions[i, 0], cmap='gray')
    axes[0, i].axis('off'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original'); axes[1, 0].set_ylabel('Reconstruida')
plt.tight_layout(); plt.show()

## Autoencoder variacional

El encoder produce una media y una varianza para el código latente. Durante el entrenamiento tomamos una muestra de esa distribución. La pérdida combina reconstrucción y una penalización que acerca los códigos a una normal estándar.

In [ ]:
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(28 * 28, 150), nn.SELU(),
            nn.Linear(150, 100), nn.SELU(),
        )
        self.mean = nn.Linear(100, latent_dim)
        self.log_variance = nn.Linear(100, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 100), nn.SELU(),
            nn.Linear(100, 150), nn.SELU(),
            nn.Linear(150, 28 * 28), nn.Sigmoid(),
        )
        self.latent_dim = latent_dim

    def encode(self, images):
        hidden = self.encoder(images)
        return self.mean(hidden), self.log_variance(hidden)

    def reparameterize(self, mean, log_variance):
        std = torch.exp(0.5 * log_variance)
        return mean + std * torch.randn_like(std)

    def forward(self, images):
        mean, log_variance = self.encode(images)
        z = self.reparameterize(mean, log_variance)
        reconstruction = self.decoder(z).view(-1, 1, 28, 28)
        return reconstruction, mean, log_variance

def vae_loss(reconstruction, images, mean, log_variance):
    reconstruction_loss = F.binary_cross_entropy(reconstruction, images, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_variance - mean.square() - log_variance.exp())
    return (reconstruction_loss + kl_loss) / images.size(0)

vae = VariationalAutoencoder(latent_dim=10).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

In [ ]:
vae_history = {'train': [], 'validation': []}
for epoch in range(1, 6):
    vae.train()
    total, count = 0.0, 0
    for batch, _ in train_loader:
        batch = batch.to(device)
        vae_optimizer.zero_grad(set_to_none=True)
        reconstruction, mean, log_variance = vae(batch)
        loss = vae_loss(reconstruction, batch, mean, log_variance)
        loss.backward()
        vae_optimizer.step()
        total += loss.item() * len(batch)
        count += len(batch)
    vae_history['train'].append(total / count)
    vae.eval()
    val_total, val_count = 0.0, 0
    with torch.no_grad():
        for batch, _ in val_loader:
            batch = batch.to(device)
            reconstruction, mean, log_variance = vae(batch)
            val_total += vae_loss(reconstruction, batch, mean, log_variance).item() * len(batch)
            val_count += len(batch)
    vae_history['validation'].append(val_total / val_count)
    print(f'Época {epoch}/5 | train={vae_history["train"][-1]:.2f} | val={vae_history["validation"][-1]:.2f}')

In [ ]:
vae.eval()
with torch.no_grad():
    original = images[:8].to(device)
    reconstruction, _, _ = vae(original)
    random_codes = torch.randn(8, vae.latent_dim, device=device)
    generated = vae.decoder(random_codes).view(-1, 1, 28, 28)
fig, axes = plt.subplots(3, 8, figsize=(12, 4.5))
for i in range(8):
    axes[0, i].imshow(original[i, 0].cpu(), cmap='gray')
    axes[1, i].imshow(reconstruction[i, 0].cpu(), cmap='gray')
    axes[2, i].imshow(generated[i, 0].cpu(), cmap='gray')
    for row in range(3): axes[row, i].axis('off')
axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('VAE')
axes[2, 0].set_ylabel('Muestra z')
plt.tight_layout(); plt.show()